# OPRB benchmark: EasyOCR (eng)

Local reproducible test-set evaluation using `.venv-easyocr` (Python 3.12 + PyTorch CUDA).

In [ ]:
import json, random, re, time, unicodedata
from collections import Counter
from pathlib import Path
import pandas as pd
import torch
from jiwer import cer, wer
from tqdm.auto import tqdm

MODEL_TAG = "EasyOCR"
MODEL_ID = "EasyOCR (en)"

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = ROOT / "test"
OUTPUT_DIR = ROOT / "result" / "easyocr_gpu"
CATEGORIES = ["text", "nontext"]
MAX_SAMPLES_PER_CATEGORY = None  # Set to e.g. 5 for smoke test, None for all 9,000 pages.
RANDOM_SEED = 42
NORMALIZE_CASE = True
SAVE_EVERY = 25
BATCH_SIZE = 64
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def collect_pairs():
    pairs = []
    for category in CATEGORIES:
        image_dir, gt_dir = DATA_ROOT / category / "img", DATA_ROOT / category / "gt"
        names = sorted({p.stem for p in image_dir.glob("*.jpg")} & {p.stem for p in gt_dir.glob("*.txt")})
        if MAX_SAMPLES_PER_CATEGORY is not None:
            names = sorted(random.Random(RANDOM_SEED).sample(names, min(len(names), MAX_SAMPLES_PER_CATEGORY)))
        for stem in names:
            parts = stem.split("__", 2)
            pairs.append({
                "category": category,
                "sample_id": stem,
                "image_path": str(image_dir / f"{stem}.jpg"),
                "gt_path": str(gt_dir / f"{stem}.txt"),
                "occlusion_type": parts[0],
                "occlusion_density": parts[1] if len(parts) > 1 else "unknown",
            })
    if not pairs:
        raise RuntimeError(f"No paired OPRB files found under {DATA_ROOT}")
    return pairs

def gt_text(path):
    with open(path, encoding="utf-8-sig", errors="replace") as f:
        return " ".join(
            line.rstrip("\r\n").split("\t")[0].strip()
            for line in f
            if "\t" in line and line.split("\t", 1)[0].strip()
        )

def norm(s):
    s = re.sub(r"\s+", " ", unicodedata.normalize("NFKC", s)).strip()
    return s.casefold() if NORMALIZE_CASE else s

def metrics(ref, hyp):
    ref, hyp = norm(ref), norm(hyp)
    if not ref:
        raise ValueError("empty ground truth")
    a, b = Counter(ref.split()), Counter(hyp.split())
    matched = sum((a & b).values())
    p = matched / sum(b.values()) if b else 0.0
    r = matched / sum(a.values()) if a else 0.0
    return {
        "cer": float(cer(ref, hyp)),
        "wer": float(wer(ref, hyp)),
        "token_precision": p,
        "token_recall": r,
        "token_f1": 2 * p * r / (p + r) if p + r else 0.0,
        "gt_words": len(ref.split()),
        "pred_words": len(hyp.split()),
    }

pairs = collect_pairs()
print(f"DATA_ROOT={DATA_ROOT} | OUTPUT_DIR={OUTPUT_DIR} | pairs={len(pairs):,}")

In [ ]:
import easyocr

USE_GPU = torch.cuda.is_available()
DEVICE_NAME = torch.cuda.get_device_name(0) if USE_GPU else "CPU"
print(f"torch={torch.__version__} | cuda={USE_GPU} | device={DEVICE_NAME} | easyocr={easyocr.__version__}")
reader = easyocr.Reader(["en"], gpu=USE_GPU, verbose=False)

In [ ]:
def transcribe_page(path):
    start = time.perf_counter()
    lines = reader.readtext(str(path), detail=0, paragraph=False, batch_size=BATCH_SIZE)
    text = " ".join(str(t).strip() for t in lines if str(t).strip())
    return text.strip(), time.perf_counter() - start, 0, False

In [ ]:
RESULTS_CSV = OUTPUT_DIR / "per_image_results.csv"
old = pd.read_csv(RESULTS_CSV).to_dict("records") if RESULTS_CSV.exists() else []
valid_ids = {p["sample_id"] for p in pairs}
rows = [r for r in old if r.get("sample_id") in valid_ids and r.get("status") == "ok"]
finished = {r["sample_id"] for r in rows}
print(f"Resume: {len(finished):,} completed | remaining: {len(pairs) - len(finished):,}")

t0 = time.perf_counter()
for i, sample in enumerate(tqdm(pairs, desc="EasyOCR OPRB"), 1):
    if sample["sample_id"] in finished:
        continue
    row = {**sample, "model_id": MODEL_ID}
    try:
        prediction, latency, generated, truncated = transcribe_page(sample["image_path"])
        reference = gt_text(sample["gt_path"])
        row.update({
            "status": "ok",
            "prediction": prediction,
            "ground_truth": reference,
            "latency_sec": latency,
            "generated_tokens": generated,
            "possibly_truncated": truncated,
            **metrics(reference, prediction),
        })
        finished.add(sample["sample_id"])
    except Exception as exc:
        row.update({"status": "error", "error": repr(exc)})
    rows.append(row)
    if i % SAVE_EVERY == 0:
        pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)

pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)
results = pd.DataFrame(rows)
ok = results[results["status"] == "ok"]
summary = {
    "model": MODEL_ID,
    "dataset": "kpurkayastha/OPRB test",
    "total_expected": len(pairs),
    "n_success": int(len(ok)),
    "n_errors": int(len(results) - len(ok)),
    "mean_latency_sec": float(ok["latency_sec"].mean()) if len(ok) else None,
    "mean_cer": float(ok["cer"].mean()) if len(ok) else None,
    "mean_wer": float(ok["wer"].mean()) if len(ok) else None,
    "mean_token_f1": float(ok["token_f1"].mean()) if len(ok) else None,
    "device": DEVICE_NAME,
    "runtime_sec_this_run": time.perf_counter() - t0,
}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
display(ok[["category", "cer", "wer", "token_f1", "latency_sec"]].describe(include="all") if len(ok) else results)